# CrowdSky client — 47 Tucanae worked example

Query CrowdSky for stacked frames of the globular cluster **47 Tucanae** (NGC 104), download one
frame and its star table, display the **green channel** of the stack, and overplot circles around
the **Gaia-matched** and **unmatched** source detections.

**About the stacked FITS.** A CrowdSky stack is a *multi-extension* FITS file:

| HDU | Name | Contents |
|-----|------|----------|
| 0 | `PRIMARY` | header only — **no image data** (`hdul[0].data is None`) |
| 1–3 | `RED`, `GREEN`, `BLUE` | the three colour planes (2-D images) |
| 4 | `STAR-TAB` | SEP detections **plus** the Gaia cross-match columns |

So the green layer is `hdul["GREEN"].data`, and the Gaia match info lives in `hdul["STAR-TAB"]`.

**Prerequisites**

```bash
pip install crowdsky-client matplotlib
export CROWDSKY_API_KEY="csk_..."   # mint at https://crowdsky.univie.ac.at -> Account
```

In [ ]:
from io import BytesIO

import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.visualization import ZScaleInterval, ImageNormalize

from crowdsky_client import Client

# 47 Tucanae / NGC 104
RA_47TUC, DEC_47TUC = 6.0224, -72.0814

client = Client()   # api_key from CROWDSKY_API_KEY; base_url defaults to production

## 1. Find frames covering 47 Tuc

`frames_for_target` cone-searches the HEALPix tiles covering the position (default radius 0.75°,
about the Seestar field of view), unions the per-tile results, and dedups by frame `id`.

In [ ]:
frames = client.frames_for_target(RA_47TUC, DEC_47TUC, radius_deg=0.9)
print(f"{len(frames)} frames cover 47 Tuc")
for f in frames[:5]:
    print(f"  id={f['id']:>6}  {f.get('filter_name','?'):>5}  "
          f"exp={f.get('total_exptime')}s  stars={f.get('n_stars_detected')}  "
          f"{f.get('date_obs_start')}")

## 2. Pick a frame

Take the deepest stack (largest cumulative exposure).

In [ ]:
if not frames:
    raise SystemExit("No frames returned — check your key and try a wider radius.")

frame = max(frames, key=lambda f: f.get("total_exptime") or 0)
frame_id = frame["id"]
print(f"Using frame {frame_id}: filter={frame.get('filter_name')} "
      f"exp={frame.get('total_exptime')}s size={frame.get('file_size_bytes')} bytes")

## 3. Download the frame; read the green layer and the star table

`download_frame` returns the full FITS as bytes, which we open straight from memory. We pull the
**green** image plane and the **STAR-TAB** columns out of their named extensions. (Use
`client.download_frame_to(frame_id, path)` to stream to disk instead.)

In [ ]:
raw = client.download_frame(frame_id)
assert raw[:6] == b"SIMPLE", "expected a FITS file"

with fits.open(BytesIO(raw)) as hdul:
    hdul.info()                                  # note: PRIMARY has no data; images are extensions
    green = np.array(hdul["GREEN"].data, dtype=float)
    tab = hdul["STAR-TAB"].data
    star_x = np.asarray(tab["x"])
    star_y = np.asarray(tab["y"])
    gaia_id = np.asarray(tab["gaia_id"])         # 0 = no Gaia match
    gaia_gmag = np.asarray(tab["gaia_gmag"])     # NaN when unmatched

print("\ngreen layer:", green.shape)
print("detections:", len(star_x))

## 4. (Optional) download the standalone "star file"

`star_data` fetches the frame's star table as JSON. Note this standalone file contains the **SEP
detections only** (position, flux, shape) — the **Gaia cross-match columns are not in the JSON**,
they live in the FITS `STAR-TAB` extension used above. We fetch it here just to show the call.

In [ ]:
stars_json = client.star_data(frame_id)
print("star-file keys:", list(stars_json.keys()))
print("n detections:", len(stars_json["stars"]))
print("per-detection fields:", sorted(stars_json["stars"][0].keys()))

## 5. Split detections into Gaia-matched vs unmatched

A detection counts as Gaia-matched when it has a non-zero `gaia_id` (equivalently, a finite
`gaia_gmag`). Unmatched detections are cosmic rays, hot pixels, blends, or genuine transients.

In [ ]:
matched = gaia_id != 0
n_match = int(matched.sum())
n_unmatch = int((~matched).sum())
print(f"{n_match} Gaia-matched, {n_unmatch} unmatched (of {len(matched)})")

## 6. Plot the green layer with circled detections

Green circles mark Gaia-matched detections, red circles the unmatched ones. The `STAR-TAB` `x`/`y`
are SEP pixel coordinates, matching `imshow(origin="lower")`.

In [ ]:
norm = ImageNormalize(green, interval=ZScaleInterval())

fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(green, origin="lower", cmap="gray", norm=norm)
ax.scatter(star_x[matched], star_y[matched], s=90, facecolors="none",
           edgecolors="lime", linewidths=1.2, label=f"Gaia-matched ({n_match})")
ax.scatter(star_x[~matched], star_y[~matched], s=90, facecolors="none",
           edgecolors="red", linewidths=1.2, label=f"unmatched ({n_unmatch})")
ax.set_title(f"47 Tuc — frame {frame_id} (green channel)")
ax.set_xlabel("x [pixels]")
ax.set_ylabel("y [pixels]")
ax.legend(loc="upper right", framealpha=0.8)
plt.tight_layout()
plt.show()

## Next steps

- Filter your query server-side: `frames_by_healpix(pix, filter_name="LP", min_exptime=120)`.
- Stream large frames to disk: `client.download_frame_to(frame_id, "frame.fits")`.
- Survey the whole sky before pulling frames: `coverage = client.sky_coverage()` (an
  `astropy.table.Table`, no key required).

See the [full documentation](https://crowdsky-client.readthedocs.io) for the complete API.